# US-6: Data Cleaning & Validation

Clean and validate the extracted SGCarMart dataset for analysis.

## Phases
1. Load raw data
2. Remove invalid rows
3. Normalize sentinel values
4. Clean & type-cast all fields
5. Validation checks
6. Data quality report
7. Export clean dataset

## Phase 1: Load Raw Data

In [1]:
import polars as pl
import json
from pathlib import Path
from datetime import datetime

RAW_FILE = Path('output/full_detail_data.csv')
CLEAN_PARQUET = Path('output/clean_detail_data.parquet')
VALIDATION_FILE = Path('output/validation_summary.json')

df_raw = pl.read_csv(RAW_FILE, infer_schema_length=0, null_values=[])
print(f'Loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print(f'Columns: {sorted(df_raw.columns)}')

Loaded: 14,325 rows x 23 columns
Columns: ['accessories', 'arf', 'car_model', 'coe', 'curb_weight', 'depreciation', 'dereg_value', 'detail_page_url', 'engine_cap', 'features', 'fuel_type', 'listing_id', 'manufactured', 'mileage', 'omv', 'owners', 'power', 'price', 'reg_date', 'road_tax', 'status', 'transmission', 'vehicle_type']


In [2]:
# Quick look at the raw data
print('=== First 3 rows ===')
df_raw.head(3).glimpse()

# Null counts per column
null_counts = df_raw.select(
    [pl.col(c).null_count().alias(c) for c in df_raw.columns]
).transpose(include_header=True, header_name='column', column_names=['null_count'])

print('\n=== Null counts ===')
print(null_counts.filter(pl.col('null_count') > 0).sort('null_count', descending=True))

=== First 3 rows ===
Rows: 3
Columns: 23
$ detail_page_url <str> 'https://www.sgcarmart.com/used-cars/info/lamborghini-gallardo-lp560-4-coe-1488131/', 'https://www.sgcarmart.com/used-cars/info/bentley-continental-gt-40a-1464426/', 'https://www.sgcarmart.com/used-cars/info/honda-civic-16a-vti-1478231/'
$ listing_id      <str> '1488131', '1464426', '1478231'
$ car_model       <str> 'Lamborghini Gallardo LP560-4', 'Bentley Continental GT 4.0A V8', 'Honda Civic 1.6A VTi'
$ depreciation    <str> '36950', '31960', '18590'
$ coe             <str> '57955', '115270', '26999'
$ reg_date        <str> '19-Jul-2011', '28-May-2013', '16-Aug-2019'
$ mileage         <str> '55000', '74202', '107000'
$ road_tax        <str> '8751', '5122', '742'
$ omv             <str> '222743', '206137', '20084'
$ arf             <str> '222743', '206137', '20118'
$ engine_cap      <str> '5204', '3993', '1597'
$ fuel_type       <str> 'Petrol', 'Petrol', 'Petrol'
$ power           <str> '412', '373', '92'
$ transmission 

## Phase 2: Remove Invalid Rows

Drop rows with null listing_id (2 completely empty rows).

In [3]:
before = df_raw.shape[0]
df = df_raw.filter(pl.col('listing_id').is_not_null())
removed = before - df.shape[0]
print(f'Rows before: {before:,}')
print(f'Removed (null listing_id): {removed}')
print(f'Rows after: {df.shape[0]:,}')

Rows before: 14,325
Removed (null listing_id): 2
Rows after: 14,323


## Phase 3: Normalize Sentinel Values

Replace literal string `'None'` with actual null across all columns.

In [4]:
# Replace 'None' string with null for all string columns
df = df.with_columns([
    pl.when(pl.col(c) == 'None')
      .then(None)
      .otherwise(pl.col(c))
      .alias(c)
    for c in df.columns
    if df[c].dtype == pl.Utf8
])

# Verify sentinel removal
none_string_counts = df.select([
    (pl.col(c) == 'None').sum().alias(c)
    for c in df.columns
    if df[c].dtype == pl.Utf8
])
total_nones = none_string_counts.select(pl.sum_horizontal(*none_string_counts.columns)).item()
print(f"Remaining 'None' strings: {total_nones}")

# Show updated null counts
null_counts = df.select(
    [pl.col(c).null_count().alias(c) for c in df.columns]
).transpose(include_header=True, header_name='column', column_names=['null_count'])

print('\n=== Null counts after sentinel normalization ===')
print(null_counts.filter(pl.col('null_count') > 0).sort('null_count', descending=True))

Remaining 'None' strings: 0

=== Null counts after sentinel normalization ===
shape: (14, 2)
┌──────────────┬────────────┐
│ column       ┆ null_count │
│ ---          ┆ ---        │
│ str          ┆ u32        │
╞══════════════╪════════════╡
│ mileage      ┆ 3153       │
│ features     ┆ 2650       │
│ accessories  ┆ 2533       │
│ power        ┆ 1956       │
│ road_tax     ┆ 1857       │
│ …            ┆ …          │
│ curb_weight  ┆ 259        │
│ omv          ┆ 74         │
│ price        ┆ 56         │
│ manufactured ┆ 24         │
│ reg_date     ┆ 22         │
└──────────────┴────────────┘


## Phase 4: Clean & Type-Cast Fields

In [5]:
# --- Integer fields ---

# listing_id: direct cast
# mileage, engine_cap: direct numeric cast
# owners: parse 'X Owners' -> X, 'More than 6' -> 7
df = df.with_columns([
    pl.col('listing_id').cast(pl.Int64),
    pl.col('mileage').cast(pl.UInt32),
    pl.col('engine_cap').cast(pl.UInt32),
    # Parse owners: extract first number from string
    pl.col('owners')
        .str.extract_all(r'\d+')
        .list.first()
        .cast(pl.UInt32),
])

print('Integer fields cast.')
print(f'  listing_id: {df["listing_id"].dtype}')
print(f'  mileage: {df["mileage"].dtype}')
print(f'  engine_cap: {df["engine_cap"].dtype}')
print(f'  owners: {df["owners"].dtype}')

Integer fields cast.
  listing_id: Int64
  mileage: UInt32
  engine_cap: UInt32
  owners: UInt32


In [6]:
# --- Float fields ---

# curb_weight: strip ' kg', remove commas, then cast
df = df.with_columns([
    pl.col('curb_weight')
        .str.replace_all(' kg', '')
        .str.replace_all(',', '')
        .cast(pl.Float64)
        .alias('curb_weight'),
])

# Direct cast for already-clean numeric strings
float_fields = ['price', 'depreciation', 'omv', 'arf', 'coe', 'road_tax', 'dereg_value', 'power']
df = df.with_columns([
    pl.col(f).cast(pl.Float64) for f in float_fields
])

print('Float fields cast.')
for f in float_fields + ['curb_weight']:
    print(f'  {f}: {df[f].dtype}')

Float fields cast.
  price: Float64
  depreciation: Float64
  omv: Float64
  arf: Float64
  coe: Float64
  road_tax: Float64
  dereg_value: Float64
  power: Float64
  curb_weight: Float64


In [7]:
# --- Date fields ---

# reg_date: parse 'DD-Mon-YYYY' (e.g., '07-Mar-2024')
df = df.with_columns([
    pl.col('reg_date')
        .str.strptime(pl.Date, '%d-%b-%Y', strict=False)
        .alias('reg_date'),
])

# manufactured: parse as integer year
df = df.with_columns([
    pl.col('manufactured').cast(pl.UInt32),
])

print('Date fields cast.')
print(f'  reg_date: {df["reg_date"].dtype}')
print(f'  manufactured: {df["manufactured"].dtype}')
print(f'\n  reg_date range: {df["reg_date"].min()} to {df["reg_date"].max()}')
print(f'  manufactured range: {df["manufactured"].min()} to {df["manufactured"].max()}')

Date fields cast.
  reg_date: Date
  manufactured: UInt32

  reg_date range: 1955-12-16 to 2026-03-31
  manufactured range: 1939 to 2026


In [8]:
# --- Categorical fields ---

# Inspect unique values before casting
cat_fields = ['fuel_type', 'transmission', 'vehicle_type', 'status']
for f in cat_fields:
    unique_vals = df[f].unique().sort().to_list()
    print(f'{f} ({len(unique_vals)} unique): {unique_vals}')

fuel_type (8 unique): ['Diesel', 'Diesel (Euro 4 Engine and Below)', 'Diesel (Euro 5 Engine and Above)', 'Diesel (Registered as Commercial Vehicle)', 'Diesel-Electric (Euro 5 Engine and Above)', 'Electric', 'Petrol', 'Petrol-Electric']
transmission (2 unique): ['Auto', 'Manual']
vehicle_type (11 unique): ['Bus/Mini Bus', 'Hatchback', 'Luxury Sedan', 'MPV', 'Mid-Sized Sedan', 'Others', 'SUV', 'Sports Car', 'Stationwagon', 'Truck', 'Van']
status (3 unique): ['Available for sale', 'CLOSED', 'SOLD']


In [9]:
# Cast to Categorical
df = df.with_columns([
    pl.col(f).cast(pl.Categorical) for f in cat_fields
])

print('Categorical fields cast.')
for f in cat_fields:
    print(f'  {f}: {df[f].dtype}')

Categorical fields cast.
  fuel_type: Categorical
  transmission: Categorical
  vehicle_type: Categorical
  status: Categorical


In [10]:
# --- Final schema ---
print('=== Final Schema ===')
for col in df.columns:
    dtype = df[col].dtype
    nulls = df[col].null_count()
    print(f'  {col:20s} {str(dtype):15s} nulls={nulls:,}')

print(f'\nFinal shape: {df.shape[0]:,} rows x {df.shape[1]} columns')

=== Final Schema ===
  detail_page_url      String          nulls=0
  listing_id           Int64           nulls=0
  car_model            String          nulls=0
  depreciation         Float64         nulls=489
  coe                  Float64         nulls=956
  reg_date             Date            nulls=22
  mileage              UInt32          nulls=3,153
  road_tax             Float64         nulls=1,857
  omv                  Float64         nulls=74
  arf                  Float64         nulls=0
  engine_cap           UInt32          nulls=977
  fuel_type            Categorical     nulls=0
  power                Float64         nulls=1,956
  transmission         Categorical     nulls=0
  manufactured         UInt32          nulls=24
  owners               UInt32          nulls=0
  dereg_value          Float64         nulls=1,188
  curb_weight          Float64         nulls=259
  status               Categorical     nulls=0
  features             String          nulls=2,650
  access

## Phase 5: Validation Checks

In [11]:
checks = {}

# AC6.1: Zero null IDs
null_ids = df.filter(pl.col('listing_id').is_null()).shape[0]
checks['zero_null_ids'] = {'pass': null_ids == 0, 'null_count': null_ids}
print(f"{'PASS' if null_ids == 0 else 'FAIL'}: Null IDs = {null_ids}")

# AC6.2: Zero negative or zero prices
bad_prices = df.filter(
    (pl.col('price').is_not_null()) & (pl.col('price') <= 0)
).shape[0]
checks['no_zero_negative_prices'] = {'pass': bad_prices == 0, 'count': bad_prices}
print(f"{'PASS' if bad_prices == 0 else 'FAIL'}: Zero/negative prices = {bad_prices}")

# AC6.3: All IDs unique
total = df.shape[0]
unique = df.select(pl.col('listing_id').n_unique()).item()
dupes = total - unique
checks['unique_ids'] = {'pass': dupes == 0, 'duplicates': dupes}
print(f"{'PASS' if dupes == 0 else 'FAIL'}: Duplicate IDs = {dupes}")

# AC6.4: Mileage >= 0
neg_mileage = df.filter(
    (pl.col('mileage').is_not_null()) & (pl.col('mileage') < 0)
).shape[0]
checks['mileage_non_negative'] = {'pass': neg_mileage == 0, 'count': neg_mileage}
print(f"{'PASS' if neg_mileage == 0 else 'FAIL'}: Negative mileage = {neg_mileage}")

# AC6.5: Registration dates valid
null_dates = df.filter(pl.col('reg_date').is_null()).shape[0]
date_min = df['reg_date'].min()
date_max = df['reg_date'].max()
checks['valid_reg_dates'] = {'pass': True, 'null_count': null_dates, 'range': f'{date_min} to {date_max}'}
print(f'PASS: reg_date nulls={null_dates}, range={date_min} to {date_max}')

all_pass = all(v['pass'] for v in checks.values())
print(f"\n{'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}")

PASS: Null IDs = 0
PASS: Zero/negative prices = 0
PASS: Duplicate IDs = 0
PASS: Negative mileage = 0
PASS: reg_date nulls=22, range=1955-12-16 to 2026-03-31

ALL CHECKS PASSED


## Phase 6: Data Quality Report

In [12]:
# Field completeness
total_rows = df.shape[0]
completeness = {
    col: {
        'total': total_rows,
        'non_null': total_rows - df[col].null_count(),
        'null': df[col].null_count(),
        'completeness_pct': round((total_rows - df[col].null_count()) / total_rows * 100, 1)
    }
    for col in df.columns
}

print('=== Field Completeness ===')
for col, info in sorted(completeness.items(), key=lambda x: x[1]['completeness_pct']):
    pct = info['completeness_pct']
    bar = '#' * int(pct / 2) + '.' * (50 - int(pct / 2))
    print(f"  {col:20s} {bar} {pct:5.1f}% ({info['non_null']:,}/{total_rows:,})")

=== Field Completeness ===
  mileage              #######################################...........  78.0% (11,170/14,323)
  features             ########################################..........  81.5% (11,673/14,323)
  accessories          #########################################.........  82.3% (11,790/14,323)
  power                ###########################################.......  86.3% (12,367/14,323)
  road_tax             ###########################################.......  87.0% (12,466/14,323)
  dereg_value          #############################################.....  91.7% (13,135/14,323)
  engine_cap           ##############################################....  93.2% (13,346/14,323)
  coe                  ##############################################....  93.3% (13,367/14,323)
  depreciation         ################################################..  96.6% (13,834/14,323)
  curb_weight          #################################################.  98.2% (14,064/14,323)
  o

In [13]:
# Numeric field summaries
numeric_cols = ['price', 'depreciation', 'omv', 'arf', 'coe', 'road_tax',
               'mileage', 'engine_cap', 'power', 'dereg_value', 'curb_weight', 'owners']

stats = df.select([
    pl.col(c).median().alias(f'{c}_median') for c in numeric_cols
] + [
    pl.col(c).mean().alias(f'{c}_mean') for c in numeric_cols
] + [
    pl.col(c).std().alias(f'{c}_std') for c in numeric_cols
] + [
    pl.col(c).min().alias(f'{c}_min') for c in numeric_cols
] + [
    pl.col(c).max().alias(f'{c}_max') for c in numeric_cols
])

print('=== Numeric Field Summary ===')
print(f"{'Field':20s} {'Min':>12s} {'Median':>12s} {'Mean':>12s} {'Max':>12s} {'Std':>12s}")
print('-' * 82)
for c in numeric_cols:
    mn = stats[f'{c}_min'].item()
    md = stats[f'{c}_median'].item()
    me = stats[f'{c}_mean'].item()
    mx = stats[f'{c}_max'].item()
    sd = stats[f'{c}_std'].item()
    fmt = lambda v: f'{v:,.0f}' if v is not None else 'N/A'
    print(f'  {c:18s} {fmt(mn):>12s} {fmt(md):>12s} {fmt(me):>12s} {fmt(mx):>12s} {fmt(sd):>12s}')

=== Numeric Field Summary ===
Field                         Min       Median         Mean          Max          Std
----------------------------------------------------------------------------------
  price                     2,000       86,000      132,675    2,788,000      173,179
  depreciation              3,260       17,240       23,637      587,360       26,746
  omv                       1,766       32,934       49,992      765,955       59,608
  arf                           0       30,297       56,321    1,754,980      103,712
  coe                          10       47,000       57,095      158,004       29,758
  road_tax                     70        1,202        1,744       12,381        1,727
  mileage                       1       85,000       85,759    1,900,000       58,332
  engine_cap                    0        1,984        2,219       15,681        1,202
  power                         0          125          159          747           96
  dereg_value              

In [14]:
# Outlier detection using IQR
print('=== Outlier Detection (IQR method) ===')
outlier_summary = {}
for c in numeric_cols:
    q1 = df[c].quantile(0.25)
    q3 = df[c].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df.filter(
        (pl.col(c).is_not_null()) &
        ((pl.col(c) < lower) | (pl.col(c) > upper))
    ).shape[0]
    non_null = total_rows - df[c].null_count()
    pct = outliers / non_null * 100 if non_null > 0 else 0
    outlier_summary[c] = {'count': outliers, 'pct': round(pct, 1), 'range': f'({lower:,.0f}, {upper:,.0f})'}
    print(f'  {c:18s} outliers={outliers:5,} ({pct:4.1f}%) IQR range={lower:,.0f} to {upper:,.0f}')

=== Outlier Detection (IQR method) ===
  price              outliers=  919 ( 6.4%) IQR range=-106,250 to 312,550
  depreciation       outliers=1,270 ( 9.2%) IQR range=1,160 to 36,120
  omv                outliers=1,416 ( 9.9%) IQR range=-17,446 to 91,526
  arf                outliers=1,408 ( 9.8%) IQR range=-48,830 to 117,846
  coe                outliers=   31 ( 0.2%) IQR range=-29,196 to 143,088
  road_tax           outliers=1,509 (12.1%) IQR range=-1,008 to 3,504
  mileage            outliers=  108 ( 1.0%) IQR range=-67,500 to 232,500
  engine_cap         outliers=  931 ( 7.0%) IQR range=4 to 3,988
  power              outliers=1,007 ( 8.1%) IQR range=-38 to 318
  dereg_value        outliers=  363 ( 2.8%) IQR range=-75,352 to 167,588
  curb_weight        outliers=  579 ( 4.1%) IQR range=730 to 2,410
  owners             outliers=    0 ( 0.0%) IQR range=-2 to 6


In [15]:
# Category distributions
print('=== Category Distributions ===')
cat_distributions = {}
for c in cat_fields:
    dist = df.group_by(c).agg(pl.len().alias('count')).sort('count', descending=True)
    cat_distributions[c] = {row[0]: row[1] for row in dist.iter_rows()}
    print(f'\n  {c}:')
    for val, cnt in cat_distributions[c].items():
        label = '(null)' if val is None else val
        print(f'    {str(label):25s} {cnt:6,}')

=== Category Distributions ===

  fuel_type:
    Petrol                     9,951
    Petrol-Electric            1,907
    Diesel                     1,159
    Electric                     977
    Diesel (Euro 5 Engine and Above)    321
    Diesel (Euro 4 Engine and Below)      4
    Diesel-Electric (Euro 5 Engine and Above)      3
    Diesel (Registered as Commercial Vehicle)      1

  transmission:
    Auto                      13,138
    Manual                     1,185

  vehicle_type:
    SUV                        3,440
    Luxury Sedan               2,597
    Sports Car                 2,336
    MPV                        1,419
    Mid-Sized Sedan            1,318
    Hatchback                  1,217
    Van                          864
    Truck                        698
    Stationwagon                 232
    Bus/Mini Bus                 142
    Others                        60

  status:
    Available for sale        14,289
    SOLD                          30
    CLOSED   

## Phase 7: Export Clean Dataset

In [16]:
# Export clean Parquet
df.write_parquet(CLEAN_PARQUET, compression='snappy')
print(f'Clean Parquet: {CLEAN_PARQUET} ({CLEAN_PARQUET.stat().st_size / 1024 / 1024:.1f} MB)')

# Verify it loads
df_verify = pl.read_parquet(CLEAN_PARQUET)
print(f'Verification: {df_verify.shape[0]:,} rows x {df_verify.shape[1]} columns')
print(f'Schema matches: {df_verify.columns == df.columns}')

Clean Parquet: output/clean_detail_data.parquet (2.1 MB)
Verification: 14,323 rows x 23 columns
Schema matches: True


In [17]:
# Export validation summary
validation_summary = {
    'generated_at': datetime.now().isoformat(),
    'source_file': str(RAW_FILE),
    'rows_after_cleaning': df.shape[0],
    'columns': df.shape[1],
    'schema': {col: str(df[col].dtype) for col in df.columns},
    'validation_checks': checks,
    'completeness': completeness,
    'outliers': outlier_summary,
    'category_distributions': cat_distributions,
}

with open(VALIDATION_FILE, 'w') as f:
    json.dump(validation_summary, f, indent=2, default=str)

print(f'Validation summary: {VALIDATION_FILE} ({VALIDATION_FILE.stat().st_size / 1024:.1f} KB)')

print(f"\n{'='*60}")
print('CLEANING COMPLETE')
print(f"{'='*60}")
print(f'  Rows: {df.shape[0]:,}')
print(f'  Columns: {df.shape[1]}')
print(f'  Output: {CLEAN_PARQUET}')
print(f'  Report: {VALIDATION_FILE}')

Validation summary: output/validation_summary.json (5.9 KB)

CLEANING COMPLETE
  Rows: 14,323
  Columns: 23
  Output: output/clean_detail_data.parquet
  Report: output/validation_summary.json
